In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
%python
dbutils.widgets.text("catalogo", "catalog_gamd")
dbutils.widgets.text("esquema_source", "silver")
dbutils.widgets.text("esquema_sink", "golden")

In [0]:
%python
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

### Load

#### kpis_products_ordered

In [0]:
%python

df_products_ordered = spark.table(f"{catalogo}.{esquema_source}.products_ordered")

In [0]:
%python

# ingestar kpis_products_ordered
df_products_ordered.write.format("delta").mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.kpis_products_ordered")

#### kpis_eval_set

In [0]:
%python

df_kpis_eval_set = spark.table(f"{catalogo}.{esquema_source}.kpis_eval_set")

In [0]:
%python

# ingestar df_kpis_eval_set
df_kpis_eval_set.write.format("delta").mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.kpis_eval_set")

### Load data into azure SQL Database

In [0]:
%python

# constantes para scope de nuestro datalake
scopesql = "scopeforSQL"
keysql = "passjdbcSQL"

In [0]:
%python
jdbcHostname = "sqlgamd.database.windows.net"
jdbcDatabase = "sqlstore"
jdbcPort = 1433
jdbcUsername = "sa_gamd"
table_products = "tbl_kpis_products_ordered"
table_eval_set = "tbl_kpis_eval_set"
jdbcPassword = dbutils.secrets.get(scope = f"{scopesql}", key = f"{keysql}")

In [0]:
%python
# URL JDBC
jdbcUrl = f"jdbc:sqlserver://{jdbcHostname}:{jdbcPort};database={jdbcDatabase};encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"

In [0]:
%python

# ingestar kpis_products_ordered

df_products_ordered.write.format("jdbc") \
    .mode("overwrite") \
    .option("url", jdbcUrl) \
    .option("dbtable", table_products) \
    .option("user", jdbcUsername) \
    .option("password", jdbcPassword) \
    .save()
       
# ngestar df_kpis_eval_set

df_products_ordered.write.format("jdbc") \
    .mode("overwrite") \
    .option("url", jdbcUrl) \
    .option("dbtable", table_eval_set) \
    .option("user", jdbcUsername) \
    .option("password", jdbcPassword) \
    .save()
       
